# Shikibu 学習（Colab 本番用）

このノートブックは、**Google Colab の L4 ランタイム**で評価関数の学習（M3）を回すためのものです。
学習ループは `python -m shikibu.train --config <TOML> [--resume]` で動きます。

使う前に次を確認してください。
- ランタイムは **GPU（L4）** を選ぶ。
- Google Drive に次を置く（場所はすべてこのノートブックの変数で決まる）。
  - `MyDrive/Shikibu/`：リポジトリ（無ければ GitHub から取得する）
  - `MyDrive/shikibu_wheels/`：`shk_feat-*.whl`（無ければビルドして保存する）
  - `MyDrive/shikibu/shards/`：学習用シャード `shard_NNNNN.psv` と、検証用の `shard_val.psv`
  - `MyDrive/shikibu/runs/<設定名>/`：学習の出力（`checkpoint.pt` と `log.jsonl`）が自動で作られる
- 学習は**途中で切れる前提**。30分ごとに `checkpoint.pt` を保存し、次のセッションでは `--resume` で続きから回す。

## 1. Google Drive をマウントする

Drive にあるリポジトリ・wheel・データ・出力を使うため、最初にマウントします。
初回は認証画面が出るので、表示に従って許可してください。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')

## 2. リポジトリを取得（または更新）する

- `REPO`：リポジトリの場所。Drive に既にあれば `git pull`、無ければ GitHub から `git clone` します。
- `BRANCH`：学習コードのブランチ（M3 の成果が入っているブランチ）。

In [ ]:
import os
import subprocess
import glob
import json

REPO_URL = "https://github.com/Harumichi-Toshiki/Shikibu.git"   # 取得元（Drive に無いとき）
BRANCH = "claude/zealous-heisenberg-heegv5"                      # 学習コードのブランチ
REPO = "/content/drive/MyDrive/Shikibu"                          # Drive 上のリポジトリ
WHEEL_DIR = "/content/drive/MyDrive/shikibu_wheels"              # wheel の保存先
DATA_DIR = "/content/drive/MyDrive/shikibu/shards"               # 学習・検証用のシャード
RUNS_DIR = "/content/drive/MyDrive/shikibu/runs"                 # 学習の出力先

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("リポジトリ:", REPO, "/ ブランチ:", BRANCH)

## 3. wheel（`shikibu_feat`）をインストールする

- Drive に `shk_feat-*.whl` があれば、それを入れるだけで済みます（数秒）。
- 無ければ Rust をインストールし、`maturin build --release` で wheel を作って Drive に保存します（初回のみ）。
- ソースを変えたら `MyDrive/shikibu_wheels` の wheel を消してから実行し直してください。

In [ ]:
# 既存の wheel を探す。複数あれば一番新しいものを使う
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)

if wheels:
    # 保存済みの wheel を入れる（ビルドは不要）
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    # Rust が無ければ入れる（ランタイムごとに一度だけ必要）
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])

    # wheel をビルドして Drive に保存し、そのまま入れる
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml",
                    "-o", WHEEL_DIR], cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして Drive に保存しました:", built[-1])

# 動くことを確認する（PyTorch は Colab に入っている）
import torch
import shikibu_feat
print("torch", torch.__version__, "/ CUDA:", torch.cuda.is_available(), "/ GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("論理コア数:", os.cpu_count(), "/ shikibu_feat OK, v0 の次元数:", shikibu_feat.num_features(0))

## 4. 設定を選ぶ（このセッションの量を決める）

- `CONFIG_NAME`：`colab_v0`（RayPair_v0）か `colab_v1`（RayPair_v1）。設定の中身は `configs/<名前>.toml`。
- `SESSION_MAX_STEPS`：**通算の更新回数**の上限（設定の `max_steps`）。`None` なら設定ファイルの値を使う。
  このセッションで区切りたい更新回数を入れる（例：`60000`）。次のセッションでは、より大きい値にして再実行する。
- 出力は `MyDrive/shikibu/runs/<CONFIG_NAME>/` に保存され、`--resume` で続きから回せます。

In [ ]:
import re

CONFIG_NAME = "colab_v0"   # "colab_v0" または "colab_v1"
SESSION_MAX_STEPS = None   # None なら設定の max_steps。数値なら通算の更新回数の上限

src = os.path.join(REPO, "configs", CONFIG_NAME + ".toml")
text = open(src, encoding="utf-8").read()
if SESSION_MAX_STEPS is not None:
    text, n = re.subn(r"^max_steps = \d+", "max_steps = %d" % SESSION_MAX_STEPS, text, flags=re.M)
    assert n == 1, "max_steps の行が見つからない"

CONFIG_PATH = "/content/run_config.toml"   # このセッション用の設定（リポジトリは書き換えない）
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(text)
OUT_DIR = os.path.join(RUNS_DIR, CONFIG_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

# データの確認：学習用シャードと検証用シャードがあるか
train_shards = sorted(glob.glob(os.path.join(DATA_DIR, "shard_[0-9]*.psv")))
val_shard = os.path.join(DATA_DIR, "shard_val.psv")
print("学習用シャード:", len(train_shards), "個 /", "検証用:", "あり" if os.path.exists(val_shard) else "なし（要配置）")
print("出力先:", OUT_DIR, "/ checkpoint:", "あり（--resume で続きから）" if os.path.exists(os.path.join(OUT_DIR, "checkpoint.pt")) else "なし（最初から）")
print(text)

## 5. 学習を実行する

- `checkpoint.pt` があれば `--resume` を付けて続きから学習します。無ければ最初から学習します。
- 設定の `max_steps` に達すると、保存して終わります（この終了はエラーではありません）。
- 途中で切れたら、このセルをもう一度実行すれば続きから再開します（最後に保存された時点から）。
- 学習の進み具合（loss・局面/秒）は `log.jsonl` に書かれ、次のセルで学習曲線を描きます。

In [ ]:
ckpt = os.path.join(OUT_DIR, "checkpoint.pt")
cmd = ["python", "-m", "shikibu.train", "--config", CONFIG_PATH]
if os.path.exists(ckpt):
    cmd.append("--resume")

env = dict(os.environ, PYTHONPATH=os.path.join(REPO, "python"))
print("実行:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO, env=env)
print("終了コード:", result.returncode)

## 6. 学習曲線を描く

`log.jsonl` から、学習の loss・学習速度（局面/秒）・検証の MSE を描きます。
Drive からのコピー速度（MB/秒）も、キャッシュの記録があれば表示します。

In [ ]:
import matplotlib.pyplot as plt

LOG = os.path.join(OUT_DIR, "log.jsonl")
recs = [json.loads(line) for line in open(LOG, encoding="utf-8") if line.strip()]
train = [r for r in recs if r["type"] == "train"]
val = [r for r in recs if r["type"] == "val"]
cache = [r for r in recs if r["type"] == "cache"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot([r["step"] for r in train], [r["loss"] for r in train])
axes[0].set_title("train loss")
axes[1].plot([r["step"] for r in train], [r["positions_per_sec"] for r in train])
axes[1].set_title("positions / sec")
axes[2].plot([r["step"] for r in val], [r["val_mse"] for r in val], marker="o")
axes[2].set_title("val MSE (sigmoid)")
for ax in axes:
    ax.set_xlabel("update step")
    ax.grid(True)
plt.tight_layout()
plt.show()

if cache:
    copied = [c for r in cache for c in r["copied"]]
    if copied:
        mbs = [c["mb_per_s"] for c in copied]
        print("Drive からのコピー: %d ファイル / 平均 %.1f MB/s（毎秒100万局面には約40MB/s が必要）" % (len(copied), sum(mbs) / len(mbs)))
    else:
        print("コピーの記録はまだありません")
print("最新の学習行:", train[-1] if train else "なし")